# RAG Embedding Generator
Runs on Colab GPU. Uploads your `.txt` files, chunks them, embeds with `all-mpnet-base-v2`, and exports a parquet you download and load into pgvector locally.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# ── 1. Install deps ───────────────────────────────────────────────────────────
!pip install -q sentence-transformers pandas pyarrow

In [ ]:
# ── 2. Upload your .txt files ─────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # select one or more .txt files from your machine
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# ── 3. Config ─────────────────────────────────────────────────────────────────
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
MODEL_NAME    = 'sentence-transformers/all-mpnet-base-v2'  # 768-dim
OUTPUT_FILE   = 'embeddings.parquet'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# ── 4. Chunk ──────────────────────────────────────────────────────────────────
def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size].strip())
        start += size - overlap
    return [c for c in chunks if c]

rows = []   # {source, content}
for filename, data in uploaded.items():
    text = data.decode('utf-8', errors='replace')
    for chunk in chunk_text(text):
        rows.append({'source': filename, 'content': chunk})

print(f'Total chunks: {len(rows)}')

In [ ]:
# ── 5. Embed ──────────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer(MODEL_NAME, device=device)

texts = [r['content'] for r in rows]
embeddings = model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f'Embedding matrix: {embeddings.shape}')

In [ ]:
# ── 6. Export to parquet ──────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(rows)
# Store each embedding as a Python list — parquet handles it fine
df['embedding'] = [e.tolist() for e in embeddings]
df.to_parquet(OUTPUT_FILE, index=False)
print(f'Saved {len(df)} rows to {OUTPUT_FILE}  ({df.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory)')

In [ ]:
# ── 7. Download ───────────────────────────────────────────────────────────────
files.download(OUTPUT_FILE)
print('Done! Move embeddings.parquet into backend/ingestion/ and run load-embeddings.py')